# Telco Customer Churn — Exploratory Data Analysis (EDA)

**Assessment:** Cognirex Solutions AI/ML Engineer Technical Assessment
**Candidate:** Jyotiprakash
**Dataset:** IBM Telco Customer Churn (`WA_Fn-UseC_-Telco-Customer-Churn.csv`)

---

## 1. Executive Summary & Objectives

The primary objective of this Exploratory Data Analysis is to uncover behavioral, transactional, and demographic drivers of customer churn within the IBM Telco Customer Churn dataset. Specifically, we investigate:
- Class imbalance and baseline churn distribution
- Integrity and missingness diagnostics, specifically `TotalCharges`
- Contract and tenure cohort dynamics
- High-risk service combinations (e.g., Fiber Optic without support services)
- Feature distributions, outliers, and rank correlations

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.churn_prediction.features import clean_total_charges, compute_tenure_group, compute_service_count
from src.churn_prediction.config import DEFAULT_DATA_PATH

sns.set_theme(style="whitegrid")
print("Libraries loaded successfully.")

## 2. Dataset Loading & Dimension Verification
We load the raw dataset and verify its shape, columns, and data types.

In [ ]:
df_raw = pd.read_csv(project_root / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(f"Dataset Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.info()

## 3. Data Integrity & Missing Values Diagnostics

Direct null counts indicate 0 missing values. However, `TotalCharges` is stored as an object data type. Further inspection reveals blank strings corresponding to new customers with `tenure == 0`.

In [ ]:
blanks = df_raw[df_raw["TotalCharges"].astype(str).str.strip() == ""]
print(f"Blank TotalCharges count: {len(blanks)}")
print(f"Associated tenure distribution: {blanks['tenure'].value_counts().to_dict()}")

# Clean TotalCharges safely using centralized transformer
df = clean_total_charges(df_raw)
print(f"Nulls after clean_total_charges: {df['TotalCharges'].isna().sum()}")
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customerIDs:", df['customerID'].duplicated().sum())

## 4. Target Class Distribution (Imbalance Analysis)
Churn distribution shows 5,174 retained customers (73.46%) vs. 1,869 churned customers (26.54%).
Because negative cases dominate by nearly 3:1, accuracy alone would be an uninformative metric.

In [ ]:
target_counts = df["Churn"].value_counts()
target_pct = df["Churn"].value_counts(normalize=True) * 100
print(pd.DataFrame({"Count": target_counts, "Percentage (%)": target_pct.round(2)}))

## 5. Feature Engineering: Tenure Cohorts & Service Count
We construct:
- `tenure_group`: 0-12 Month, 13-24 Month, 25-48 Month, 49-72 Month cohorts
- `service_count`: total active subscribed services (excluding 'No internet service' / 'No phone service')

In [ ]:
df["tenure_group"] = compute_tenure_group(df["tenure"])
df["service_count"] = compute_service_count(df)
df["Churn_binary"] = (df["Churn"] == "Yes").astype(int)

print("Service count distribution:")
print(df["service_count"].value_counts().sort_index())

## 6. Key Bivariate Patterns & Business Insights

### 6.1 Churn by Contract Type
Month-to-month contracts have a **42.7%** churn rate, compared to **11.3%** for One year and **2.8%** for Two year contracts.

In [ ]:
contract_churn = df.groupby("Contract")["Churn_binary"].agg(["count", "mean"]).reset_index()
contract_churn["churn_pct"] = contract_churn["mean"] * 100
print(contract_churn)

### 6.2 Churn by Tenure Cohort
New customers in their first year (0-12 months) experience a **47.7%** churn rate, whereas long-tenure customers (49-72 months) have a low churn rate of **9.5%**.

In [ ]:
tenure_churn = df.groupby("tenure_group")["Churn_binary"].agg(["count", "mean"]).reset_index()
tenure_churn["churn_pct"] = tenure_churn["mean"] * 100
print(tenure_churn)

### 6.3 Internet Service & Support Services
Fiber optic customers have high churn (~41.9%), driven primarily by absence of Tech Support and Online Security.

In [ ]:
internet_support = df.groupby(["InternetService", "TechSupport"])["Churn_binary"].agg(["count", "mean"]).reset_index()
internet_support["churn_pct"] = internet_support["mean"] * 100
print(internet_support)

## 7. Correlation Analysis
Spearman rank correlation demonstrates that `tenure` has a strong negative association with churn (-0.37), while `MonthlyCharges` has a positive correlation (+0.18).

In [ ]:
corr = df[["tenure", "MonthlyCharges", "TotalCharges", "service_count", "Churn_binary"]].corr(method="spearman")
print(corr.round(3))